In [0]:
import pandas as pd

df = pd.read_excel('/Workspace/Users/ananthagiri_abhiram@next.co.uk/iAudit_Shaloob/email_audit/Email Assist vul eod.xlsx', 'Vul selected')

In [0]:
df.head(3)

,intent_confidence,intent_result,time,ticketid,Agent Email id,vulnerable_result,vulnerable_confidence,agent_input
0,High,Enquiry,2026-05-30 10:41:00,29890394,NaN,Vulnerable,High,Subject: Cancel Item(s)\n=====================...
1,High,Enquiry,2026-05-30 23:36:00,29893505,NaN,Vulnerable,High,Subject: An online order for fashion items or ...
2,High,Complaint,2026-05-30 09:16:00,29897718,NaN,Vulnerable,High,Subject: Order not delivered\n================...


In [0]:
df = df.rename(columns={'ticketid': 'Ticket Id'})

#Vul

In [0]:
# vul_selected = df[df['align'].str.lower() == 'no']
vul_selected = df.copy()

# dfvul = vul_selected[['Ticket Id','agent_input','Agent Email Address']].reset_index(drop=True).rename(columns={'Ticket Id':'id','agent_input':'convo','Agent Email Address':'agent_id'})
dfvul = vul_selected[['Ticket Id','agent_input']].reset_index(drop=True).rename(columns={'Ticket Id':'id','agent_input':'convo','Agent Email Address':'agent_id'})
print(dfvul.shape)
dfvul = dfvul.drop_duplicates()
print(dfvul.shape)

# dfvul

# Assuming you want to search for the 'id' values from dfvul in the contactcentre_prod.staging.zen_live9 table

id_list = dfvul['id'].tolist()
query = f"""
SELECT id, conversation_id, Agent_EmailAddress AS agent_id
FROM contactcentre_prod.staging.zen_live9
WHERE id IN ({','.join([f"'{x}'" for x in id_list])})
"""
logger_df = spark.sql(query).toPandas().drop_duplicates()


(600, 2)
(600, 2)


In [0]:
len(set(dfvul[dfvul['id']==30039921]['convo'].to_list()))

0

In [0]:
logger_df.id.nunique()

571

In [0]:
# dfvul

In [0]:
dfvul['id'].nunique(), 

(600,)

In [0]:
dfvul['id'] = dfvul['id'].astype(str)
logger_df['id'] = logger_df['id'].astype(str)
# dfvul2 = dfvul.merge(logger_df, on=['id','agent_id'], how='inner')
dfvul2 = dfvul.merge(logger_df, on=['id'], how='inner')
dfvul2 = dfvul2.drop_duplicates()
dfvul2 = dfvul2[['id','conversation_id','agent_id','convo']]
dfvul2[['id', 'conversation_id', 'agent_id']] = dfvul2[['id', 'conversation_id', 'agent_id']].fillna('UNKNOWN')
dfvul2['pk'] = dfvul2['id'] + "|" + dfvul2['conversation_id'] + "|" + dfvul2['agent_id']

In [0]:
dfvul2

,id,conversation_id,agent_id,convo,pk
0,29890394,058c3e71-5961-42d1-9a52-ce1fc3466b6d,emma_slaney@next.co.uk,Subject: Cancel Item(s)\n=====================...,29890394|058c3e71-5961-42d1-9a52-ce1fc3466b6d|...
1,29893505,a61218b7-8ab3-4106-b4fe-6e9249de8bc4,jasmine_paton@next.co.uk,Subject: An online order for fashion items or ...,29893505|a61218b7-8ab3-4106-b4fe-6e9249de8bc4|...
2,29897718,4782580e-c0c1-4db1-8c37-4d40d25ab76b,ashish_tale@next.co.uk,Subject: Order not delivered\n================...,29897718|4782580e-c0c1-4db1-8c37-4d40d25ab76b|...
3,29905328,99a940af-6281-4e15-909e-dbeed735615d,jamie_cox@next.co.uk,Subject: Missing Payment\n====================...,29905328|99a940af-6281-4e15-909e-dbeed735615d|...
4,29909015,11639f3f-c05e-469c-b67f-bf87cf124647,claire_warley1@next.co.uk,Subject: Returns\n============================...,29909015|11639f3f-c05e-469c-b67f-bf87cf124647|...
...,...,...,...,...,...
566,30148103,4ed505a3-9664-4ead-81ba-abf77dcd0600,UNKNOWN,Subject: An online order for fashion items or ...,30148103|4ed505a3-9664-4ead-81ba-abf77dcd0600|...
567,30148118,0dae314e-9be2-4b06-9d01-f88731c00f61,UNKNOWN,Subject: First Proposal of Repayment for Accou...,30148118|0dae314e-9be2-4b06-9d01-f88731c00f61|...
568,30148134,93922027-5195-463d-92af-c223d409fc22,UNKNOWN,Subject: First Proposal of Repayment for Accou...,30148134|93922027-5195-463d-92af-c223d409fc22|...
569,30148177,e1bac49b-5440-44d6-abb3-4f2dd6e1118c,UNKNOWN,Subject: First Proposal of Repayment for Accou...,30148177|e1bac49b-5440-44d6-abb3-4f2dd6e1118c|...


In [0]:
from chat_vulnerability_main import *

In [0]:
op = vulnerability_main(dfvul2)

[2026-06-05 12:19:42] INFO - iAudit Chat - df: (571, 10)
[2026-06-05 12:19:42] INFO - iAudit Chat - Number of prompts sent to process: 4
[2026-06-05 12:19:42] INFO - iAudit Chat - Applying batch request model: databricks-meta-llama-3-3-70b-instruct


close_keyword_present:  (4, 12)


[2026-06-05 12:19:46] INFO - iAudit Chat - Number of prompts sent to process: 418
[2026-06-05 12:19:46] INFO - iAudit Chat - Applying batch request model: databricks-meta-llama-3-3-70b-instruct


30


[2026-06-05 12:26:12] INFO - iAudit Chat - Applying batch request model: databricks-meta-llama-3-3-70b-instruct
[2026-06-05 12:26:39] INFO - iAudit Chat - No of Store Accounts: 3


In [0]:
op.to_excel('../temp/email_audit_vul_Jun5.xlsx')

In [0]:
expected_cols = ['pk','id','conversation_id','agent_id','convo']


# EOD

In [0]:
import pandas as pd

df = pd.read_excel('/Workspace/Users/ananthagiri_abhiram@next.co.uk/iAudit_Shaloob/email_audit/Email Assist vul eod.xlsx', 'EOD selected')
df = df.rename(columns={'ticketid': 'Ticket Id'})

In [0]:
df

# maskeod= (df['align.1'].str.lower()=='no') & (df['Email Assist : Contact Type Result'].isin(['EOD','Complaint']))

# eod_selected = df[maskeod].reset_index(drop=True)
eod_selected = df.copy()

# dfeod = eod_selected[['Ticket Id','agent_input','Agent Email Address']].reset_index(drop=True).rename(columns={'Ticket Id':'id','agent_input':'convo','Agent Email Address':'agent_id'})
dfeod = eod_selected[['Ticket Id','agent_input']].reset_index(drop=True).rename(columns={'Ticket Id':'id','agent_input':'convo','Agent Email Address':'agent_id'})
print(dfeod.shape)

id_list = dfeod['id'].tolist()
query = f"""
SELECT id, conversation_id, Agent_EmailAddress AS agent_id
FROM contactcentre_prod.staging.zen_live9
WHERE id IN ({','.join([f"'{x}'" for x in id_list])})
"""
logger_df = spark.sql(query).toPandas().drop_duplicates()


(1115, 2)


In [0]:
dfeod['id'] = dfeod['id'].astype(str)
logger_df['id'] = logger_df['id'].astype(str)
# dfeod2 = dfeod.merge(logger_df, on=['id','agent_id'], how='inner')
dfeod2 = dfeod.merge(logger_df, on=['id'], how='inner')
dfeod2 = dfeod2.drop_duplicates()
dfeod2 = dfeod2[['id','conversation_id','agent_id','convo']]
dfeod2[['id', 'conversation_id', 'agent_id']] = dfeod2[['id', 'conversation_id', 'agent_id']].fillna('UNKNOWN')
dfeod2['pk'] = dfeod2['id']  + "|"+dfeod2['conversation_id']  + "|"+dfeod2['agent_id']

In [0]:
# dfeod2

In [0]:
from chat_eod_main import *

callids = dfeod2['conversation_id'].unique()

zenlivequery = f"""
SELECT id, CallType AS Logged_CallType, conversation_id
FROM contactcentre_prod.staging.zen_live9
WHERE conversation_id IN ({', '.join(["'" + str(i) + "'" for i in callids if pd.notnull(i)])})
"""

df_logger = spark.sql(zenlivequery).toPandas().drop_duplicates(subset=['conversation_id'])

output = complaint_eod_main(dfeod2, df_logger)

[2026-06-05 12:51:50] INFO - iAudit Chat - Number of prompts sent: 1014 to model databricks-meta-llama-3-3-70b-instruct
[2026-06-05 12:51:50] INFO - iAudit Chat - Processing batch 1 of 1 (prompts 0 to 1013)
[2026-06-05 13:10:34] INFO - iAudit Chat - Completed batch 1: 1014 responses generated
[2026-06-05 13:10:34] INFO - iAudit Chat - Generated 1014 LLM responses


Negative Pattern not mad
Negative Pattern no bother
Negative Pattern no fault
Negative Pattern not fault
Negative Pattern no compensation
Negative Pattern not so bad
Negative Pattern no complaint
Negative Pattern no complaint


[2026-06-05 13:10:51] INFO - iAudit Chat - Number of prompts sent: 863 to model databricks-meta-llama-3-3-70b-instruct
[2026-06-05 13:10:51] INFO - iAudit Chat - Processing batch 1 of 1 (prompts 0 to 862)
[2026-06-05 13:32:10] INFO - iAudit Chat - Completed batch 1: 863 responses generated
[2026-06-05 13:32:10] INFO - iAudit Chat - Generated 863 LLM responses
[2026-06-05 13:32:10] INFO - iAudit Chat - Number of prompts sent: 747 to model databricks-meta-llama-3-3-70b-instruct
[2026-06-05 13:32:10] INFO - iAudit Chat - Processing batch 1 of 2 (prompts 0 to 699)
[2026-06-05 15:01:41] INFO - iAudit Chat - Completed batch 1: 700 responses generated
[2026-06-05 15:01:41] INFO - iAudit Chat - Processing batch 2 of 2 (prompts 700 to 746)
[2026-06-05 15:09:11] INFO - iAudit Chat - Completed batch 2: 47 responses generated
[2026-06-05 15:09:11] INFO - iAudit Chat - Generated 747 LLM responses
[2026-06-05 15:09:13] WARNING - iAudit Chat - complaint_type_cc not in df_clf.columns


In [0]:
output.to_excel('../temp/email_audit_eod_Jun5.xlsx')